\newpage

## Introduction

Large AFL crowds leaving the MCG must clear through three rail corridors of widely differing capacity: Jolimont (2 platforms), Richmond (10 platforms), and Flinders Street (13 platforms). Because AFL supporter bases tend to be geographically biased, the demand placed on each corridor depends heavily on which teams are playing. This project therefore asks how much these demographics alone can drive uneven platform crowding and incomplete clearance after 90 minutes. Assignments 1-3 conceptualised, formalised, and implemented a coupled ODE system for this purpose. This report finalises the model, corrects for the inconsistencies identified in earlier feedback, tests the model's behaviour across all fixtures, and assesses the limitations of its conclusions.

## Background

Platform throughput falls sharply once pedestrian density exceeds a critical level, which Fruin (1971) places at around $2~\text{pax/m}^2$, and queueing systems demonstrate a similar saturating response as capacity is reached by demand. The model borrows these ideas in the form of a log-logistic curve (a four-parameter logistic, 4PL, with its upper asymptote fixed at 1, leaving three free parameters), adapted from dose-response modelling in pharmacology (Ritz et al., 2015). This curve represents the smooth degradation of train boarding efficiency as platform density rises, while a logistic gate throttle represents Myki gate saturation at the station entrance. Corridor demand is estimated from historical AFL attendance and supporter geography, since Melbourne's highly radial network ties each supporter base to a particular station, with limited practical alternatives.

## Methods
### State variables and governing equations

Three corridors $i \in \{J, R, F\}$ (Jolimont, Richmond, Flinders Street) are each tracked over $t \in [0, 90]$ minutes by three state variables, all in passengers (pax): remaining crowd $M_i(t)$, platform/gate queue $Q_i(t)$, and cumulative cleared passengers $C_i(t)$. These evolve as

$$\frac{dM_i}{dt} = -I_i(t), \qquad \frac{dQ_i}{dt} = I_i(t) - O_i(t), \qquad \frac{dC_i}{dt} = O_i(t)$$

where $I_i$ and $O_i$ are inflow and outflow rates (pax/min). The total $M_i + Q_i + C_i$ is conserved exactly, which is used below as a correctness check.

### Inflow and outflow dynamics

Inflow follows a Gamma-shaped pedestrian surge with density $f_i(t)$ and cumulative distribution $F_i(t)$, with mean equal to each corridor's walking time $\tau_i$ and peak slightly earlier at $\tfrac{k-1}{k}\tau_i$, gated by a logistic throttle $\Psi_i(Q_i)$ representing gate saturation. Fans still outside the station arrive at the Gamma hazard rate $h_i(t) = f_i(t) / \big(1 - F_i(t)\big)$:

$$I_i(t) = M_i(t) \cdot h_i(t) \cdot \Psi_i(Q_i), \qquad \Psi_i(Q_i) = \frac{1}{1 + \exp\big(\beta_q (Q_i - Q_{i,\text{crit}})\big)}$$

With the gate fully open ($\Psi_i = 1$), $M_i(t) = M_i(0)\big(1 - F_i(t)\big)$ and inflow reduces exactly to the unthrottled surge $M_i(0) \cdot f_i(t)$. When the gate chokes, held-back fans stay in $M_i$ and keep arriving as the queue eases. The earlier formulation, $I_i(t) = M_i(0) \cdot f_i(t) \cdot \Psi_i(Q_i)$, instead tied arrivals to the original surge timetable, so fans throttled at the gate were never re-admitted and silently dropped out of the model.

Outflow is limited by platform density $D_i = Q_i / A_i$ through a log-logistic boarding-efficiency curve (a 4PL with its upper asymptote fixed at 1) $\eta(D_i)$, bounded above by the nominal train capacity $C_{\text{base},i} = f_{\text{train},i} \cdot \text{Cap}_{\text{train}}$:

$$O_i(t) = C_{\text{base},i} \cdot \eta(D_i), \qquad \eta(D_i) = \eta_{\min} + \frac{1 - \eta_{\min}}{1 + (D_i / D_{\text{crit}})^{\gamma}}$$

$O_i(t)$ above is defined purely as a rate (pax/min). The initial formulation had $O_i(t) = \min\big(Q_i,\, C_{\text{base},i}\cdot\eta(D_i)\big)$, comparing a value ($Q_i$, pax) directly against a rate (pax/min), which has been removed. Over-depletion of the queue within a single simulation step is instead prevented in the discrete scheme.

### Parameterisation

Corridor demand shares $\alpha_i$ blend the home and away team's supporter geography, $\alpha_i = \tfrac{1}{2}(\alpha_{i,\text{home}} + \alpha_{i,\text{away}})$, with $M_i(0) = \alpha_i N_{\text{total}}$, $N_{\text{total}} = 60{,}000$ (30,000 rail-bound fans per club). Physical station parameters (Table 1), global parameters (Table 2) and team demographics (Table 3) are unchanged from the previously finalised design, aside from Jolimont's and Flinders Street's train frequencies which disagreed between the written table and the code, and Essendon's Richmond/Flinders split, which was coded based on the drafting table as opposed to the final one. `match_allocation` also keeps the rivalry flag from Part 3, which adds 10,000 fans per club for rivalry fixtures ($N_\text{total} = 80{,}000$). It is not used below: scaling attendance at a fixed demographic split deepens the surge and queues but does not change which corridor throttles first.

**Table 1: physical station parameters**

| Station $i$ | $\tau_i$ (min) | $Q_{i,\text{crit}}$ (pax) | $A_i$ (m²) | $f_{\text{train},i}$ (trains/min) |
|---|---|---|---|---|
| Jolimont ($J$) | 7.0 | 4,000 | 1,800 | 0.4 |
| Richmond ($R$) | 9.0 | 18,000 | 9,000 | 0.8 |
| Flinders St ($F$) | 18.0 | 25,000 | 15,000 | 1.0 |

**Table 2: global model parameters**

| Symbol | Meaning | Value |
|---|---|---|
| $k$ | Gamma shape | 4.0 |
| $\beta_q$ | Gate throttle sensitivity | 0.002 pax$^{-1}$ |
| $D_{\text{crit}}$ | Critical boarding density | 2.25 pax/m² |
| $\gamma$ | Boarding-efficiency curvature | 4.0 |
| $\eta_{\min}$ | Efficiency floor | 0.35 |
| $\text{Cap}_{\text{train}}$ | Train crush capacity | 1,000 pax/train |

**Table 3: corridor demographic shares by team**

| Team | $\alpha_J$ | $\alpha_R$ | $\alpha_F$ |
|---|---|---|---|
| Collingwood | 0.70 | 0.15 | 0.15 |
| Hawthorn | 0.10 | 0.70 | 0.20 |
| Melbourne | 0.20 | 0.30 | 0.50 |
| Richmond | 0.22 | 0.72 | 0.06 |
| Essendon | 0.06 | 0.68 | 0.26 |
| Carlton | 0.16 | 0.16 | 0.68 |
| St Kilda | 0.06 | 0.68 | 0.26 |
| Geelong | 0.06 | 0.33 | 0.61 |

### Visual representation

![diagram](assets/diagram.png){width=75% fig-align="center"}

Blue rectangles are state variables, pink ovals are parameters and auxiliary functions, and green ovals are flow rates. Total attendance is split by corridor into initial allocations $M_i$. Inflow fills each station's queue and is itself throttled by a balancing loop (queues choke their own further entry), and outflow drains the queue but is coupled to a platform congestion loop, where rising density degrades boarding efficiency and therefore the clearance rate.

### Implementation

The implementation in `model.py` mirrors the maths above directly, and is largely unchanged apart from the dimension, parameter and inflow corrections mentioned above.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import itertools
import numpy as np
import matplotlib.pyplot as plt
import model

os.makedirs("assets", exist_ok=True)

### Testing

Spot tests check each function at its known reference points, followed by validation guards, before the model is tested as a whole further below.

In [2]:
print("Gamma density at Jolimont's peak (t = 5.25 min):", model.gamma_density(5.25, 4.0, 1.75))
print("Gamma density at t = 60 min (long after the surge):", model.gamma_density(60.0, 4.0, 1.75))
print("Gamma hazard at t = 60 min (expect approaching 1/scale = 0.571 from below):", model.gamma_hazard(60.0, 4.0, 1.75))
print()
print("Throttle at Q = 0 (expect ~1.0):", model.queue_throttle(0.0, 4000.0, 0.002))
print("Throttle at Q = gate_crit (expect 0.5):", model.queue_throttle(4000.0, 4000.0, 0.002))
print("Throttle at Q >> gate_crit (expect ~0.0):", model.queue_throttle(9000.0, 4000.0, 0.002))
print()
print("Boarding efficiency at D ~ 0 (expect ~1.0):", model.boarding_efficiency(0.0001, 2.25, 4.0, 0.35))
print("Boarding efficiency at D = D_crit (expect ~0.675):", model.boarding_efficiency(2.25, 2.25, 4.0, 0.35))
print("Boarding efficiency at D >> D_crit (expect -> eta_min = 0.35):", model.boarding_efficiency(20.0, 2.25, 4.0, 0.35))

Gamma density at Jolimont's peak (t = 5.25 min): 0.128023890088793
Gamma density at t = 60 min (long after the surge): 4.943719068916771e-12
Gamma hazard at t = 60 min (expect approaching 1/scale = 0.571 from below): 0.5229256304541988

Throttle at Q = 0 (expect ~1.0): 0.9996646498695336
Throttle at Q = gate_crit (expect 0.5): 0.5
Throttle at Q >> gate_crit (expect ~0.0): 4.5397868702434395e-05

Boarding efficiency at D ~ 0 (expect ~1.0): 1.0
Boarding efficiency at D = D_crit (expect ~0.675): 0.675
Boarding efficiency at D >> D_crit (expect -> eta_min = 0.35): 0.3501041007567123


In [3]:
for label, func in [
    ("negative queue", lambda: model.queue_throttle(-1.0, 4000.0, 0.002)),
    ("zero gate_capacity", lambda: model.queue_throttle(100.0, 0.0, 0.002)),
    ("eta_min out of range", lambda: model.boarding_efficiency(1.0, 2.25, 4.0, 1.5)),
]:
    try:
        func()
        print(label, "-> NO ERROR RAISED")
    except ValueError as exc:
        print(label, "-> correctly raised:", exc)

negative queue -> correctly raised: queue must not be negative
zero gate_capacity -> correctly raised: gate_capacity must be a finite positive number
eta_min out of range -> correctly raised: eta_min must be greater than 0 and at most 1


## Results
### Baseline fixture

As a first check, the model is run on a single fixture (Richmond v Hawthorn) and its mass-conservation property is verified directly, before testing all fixtures at once below.

In [4]:
attendance, shares = model.match_allocation("Richmond", "Hawthorn")
times, M, Q, C = model.simulate_egress(attendance, shares)

total_mass = M + Q + C
drift = np.max(np.abs(total_mass - total_mass[0]))
print(f"Max mass-conservation drift: {drift:.2e} pax (should be ~0)")
print(f"Total cleared by t=90 (Richmond v Hawthorn): {C[-1].sum():.0f} / {attendance:.0f} pax "
      f"({100*C[-1].sum()/attendance:.2f}%)")
print(f"Breakdown by station: J={C[-1,0]:.0f}, R={C[-1,1]:.0f}, F={C[-1,2]:.0f}")

Max mass-conservation drift: 4.37e-11 pax (should be ~0)
Total cleared by t=90 (Richmond v Hawthorn): 60000 / 60000 pax (100.00%)
Breakdown by station: J=9600, R=42600, F=7800


In [5]:
figs = model.plot_state_trajectories(times, M, Q, C, "Richmond v Hawthorn")
for fig, state in zip(figs, ["remaining", "queue", "cleared"]):
    fig.savefig(f"assets/baseline_{state}.png", dpi=300)
    plt.close(fig)

![](assets/baseline_remaining.png){width=75% fig-align="center"}

![](assets/baseline_queue.png){width=75% fig-align="center"}

![](assets/baseline_cleared.png){width=75% fig-align="center"}

The baseline fixture allocates 71% of the crowd to Richmond. Richmond's queue overshoots its gate-critical threshold around the 9-minute mark, peaking near 18,900 pax at 15 minutes, and the entry throttle chokes to about a seventh of capacity ($\Psi_R \approx 0.15$). The held-back fans wait outside rather than leaving and enter as the queue drains, so all 42,600 of Richmond's allocation is cleared by $t=90$. Jolimont's queue briefly passes its own threshold, peaking near 4,200 pax, and Flinders Street, which is barely loaded in this fixture, never forms a queue. All 60,000 fans clear within the window, 95% of them by 61 minutes, and mass is conserved to within $5\times10^{-11}$ pax.

### Testing across many fixtures

One fixture shows little about the model's general behaviour, so the model is run across all 28 distinct pairings of the eight parameterised teams at the same baseline attendance. Because most fixtures clear completely, the time taken to clear 95% of the crowd ($t_{95}$) is reported alongside the fraction cleared by $t=90$.

In [6]:
teams = list(model.TEAM_SHARES)
results = []
max_drift = 0.0
max_flinders_queue = 0.0
for home, away in itertools.combinations(teams, 2):  # every distinct pairing
    att, sh = model.match_allocation(home, away)
    t_s, M_s, Q_s, C_s = model.simulate_egress(att, sh)
    cleared_fraction = C_s[-1].sum() / att
    mass = M_s + Q_s + C_s
    max_drift = max(max_drift, np.max(np.abs(mass - mass[0])))
    max_flinders_queue = max(max_flinders_queue, Q_s[:, model.STATION_NAMES.index("F")].max())
    results.append((home, away, cleared_fraction, sh["J"], sh["R"], model.time_to_clear(t_s, C_s, att)))

fractions = np.array([r[2] for r in results])
j_shares = np.array([r[3] for r in results])

print(f"Fixtures tested: {len(results)} (every distinct pairing of the {len(teams)} parameterised teams)")
print(f"Clearance fraction range: {fractions.min()*100:.2f}% - {fractions.max()*100:.2f}% "
      f"(mean {fractions.mean()*100:.2f}%)")
print(f"Max mass-conservation drift across all 28 fixtures: {max_drift:.2e} pax")
print(f"Peak Flinders Street queue across all fixtures: {max_flinders_queue:.1f} pax")

full = [r for r in results if not np.isnan(r[5])]
t95 = np.array([r[5] for r in full])
print(f"Fixtures reaching 95% cleared within 90 min: {len(full)}, t95 {t95.min():.0f} - {t95.max():.0f} min, "
      f"r with Richmond share {np.corrcoef([r[4] for r in full], t95)[0,1]:.2f}")

worst = min(results, key=lambda r: r[2])
fastest = min(full, key=lambda r: r[5])
print(f"Worst clearance: {worst[0]} v {worst[1]} ({worst[2]*100:.2f}%, Jolimont share {worst[3]*100:.0f}%)")
print(f"Fastest clearance: {fastest[0]} v {fastest[1]} (95% cleared by {fastest[5]:.0f} min, Jolimont share {fastest[3]*100:.0f}%)")

Fixtures tested: 28 (every distinct pairing of the 8 parameterised teams)
Clearance fraction range: 84.06% - 100.00% (mean 97.56%)
Max mass-conservation drift across all 28 fixtures: 1.16e-10 pax
Peak Flinders Street queue across all fixtures: 10459.4 pax
Fixtures reaching 95% cleared within 90 min: 21, t95 33 - 61 min, r with Richmond share 0.85
Worst clearance: Collingwood v Richmond (84.06%, Jolimont share 46%)
Fastest clearance: Essendon v Carlton (95% cleared by 33 min, Jolimont share 11%)


In [7]:
fig2 = model.plot_clearance_vs_jolimont(j_shares, fractions)
fig2.savefig("assets/clearance_vs_jolimont.png", dpi=300)
plt.close(fig2)

![](assets/clearance_vs_jolimont.png){width=75% fig-align="center"}

In [8]:
print(f"Correlation between Jolimont share and clearance fraction (r value): {np.corrcoef(j_shares, fractions)[0,1]:.2f}")

Correlation between Jolimont share and clearance fraction (r value): -0.91


Clearance ranged from 84.1% (Collingwood v Richmond) to 100%, averaging 97.6%, and mass was conserved to essentially 0 pax in every case. Only the seven Collingwood fixtures fail to clear within 90 minutes, and in each the shortfall is fans still queued at, or waiting outside, Jolimont. The other 21 fixtures clear completely, taking between 33 and 61 minutes to clear 95% of the crowd, with Richmond's share setting how long ($r = 0.85$). Flinders Street's queue peaks at about 10,500 pax, well under its 25,000 pax gate threshold, so it clears its full allocation in every fixture and only Jolimont and Richmond ever constrain clearance.

Plotted against each fixture's Jolimont share (instead of the largest share held by any one corridor), clearance falls steeply ($r = -0.91$). Jolimont's two-platform footprint gives it by far the smallest train capacity (400 pax/min) of the three corridors, so once a large share of the crowd is routed through it, its trains, further slowed by platform crowding, cannot carry everyone within the window. Collingwood's heavy inner-north supporter base ($\alpha_J=0.70$) is the only one that routes 38% or more of the crowd through Jolimont, and the seven Collingwood fixtures are exactly the seven that fail to clear. The relationship is therefore a Collingwood effect rather than a smooth trend: among the other 21 fixtures, Jolimont share has no bearing on clearance, since all of them clear completely.

### Targeted Jolimont services

Since only the Collingwood fixtures fail to clear, the obvious fix is to add Jolimont services for those fixtures alone. For each fixture that does not fully clear, the smallest Jolimont dispatch rate (in steps of 0.05 trains/min) that clears 99.9% of the crowd by $t=90$ (treated as clearing fully from here on) is found, and compared against the alternative of doubling Jolimont's gate threshold to 8,000 pax.

In [9]:
policy = []
for home, away, cleared, *_ in results:
    if cleared >= 0.999:
        continue
    policy.append((home, away, cleared, model.required_frequency(home, away),
                   model.fixture_clearance(home, away, {"J": {"gate_crit": 8000.0}})))

print(f"Fixtures not fully clearing at Jolimont's baseline 0.4 trains/min: {len(policy)} of {len(results)}")
print(f"{'Fixture':26} {'cleared':>8} {'J trains/min needed':>20} {'cleared, J gate 8,000':>22}")
for home, away, cleared, need, gate in policy:
    print(f"{home + ' v ' + away:26} {cleared*100:7.1f}% {need:20.2f} {gate*100:21.1f}%")

Fixtures not fully clearing at Jolimont's baseline 0.4 trains/min: 7 of 28
Fixture                     cleared  J trains/min needed  cleared, J gate 8,000
Collingwood v Hawthorn        91.4%                 0.50                  83.5%
Collingwood v Melbourne       85.2%                 0.55                  78.3%
Collingwood v Richmond        84.1%                 0.55                  77.3%
Collingwood v Essendon        94.5%                 0.45                  85.6%
Collingwood v Carlton         87.6%                 0.50                  80.4%
Collingwood v St Kilda        94.5%                 0.45                  85.6%
Collingwood v Geelong         94.5%                 0.45                  85.6%


Every fixture that fails to clear involves Collingwood, and each clears fully once Jolimont's dispatch rate rises from 0.4 to between 0.45 and 0.55 trains/min, roughly three to nine extra trains per hour, with Collingwood v Melbourne and Collingwood v Richmond needing the most. Widening Jolimont's gates instead makes every one of these fixtures worse, since admitting fans faster only moves the crowd onto an already congested platform, where boarding efficiency collapses.

## Discussion

As expected, incomplete clearance is driven by each corridor's load relative to its capacity. Jolimont, being far smaller and with fewer trains than Richmond and Flinders Street, has significant bottlenecking effects even with moderate demographic biases towards it. This is a plausible account of real-world congestion, as fans funnelled towards a small, low-capacity station can bottleneck even with substantial capacity elsewhere. 

The model assumes that fans who reach a gate mid-throttle wait outside for that same corridor instead of diverting to another station. This is a reasonable first approximation given Melbourne's limited realistic alternatives, but it remains unverified against real data, and the earlier formulation, in which these fans silently dropped out, showed how strongly clearance figures depend on it. The three corridors are also fully independent in the model, ignoring physical interchanges and relationships between them, particularly between Flinders Street and the City Loop. Walking times, gate capacities and platform areas are estimated from maps and public timetables rather than measured directly, so absolute clearance percentages should be read as indicative rather than precise.

Every fixture uses 60,000 rail-bound fans split evenly between the two clubs, so the effect of attendance, including the unexercised rivalry flag, remains untested. Only 8 of the 18 AFL clubs are parameterised, and since each appears in 7 of the 28 fixtures, the fixtures are not independent of one another and the correlation figures should be read cautiously. The corridor shares themselves come from supporter heartlands rather than measured travel data, with Essendon and St Kilda sharing identical splits, and each parameter is fixed at a single value despite plausible ranges existing for most, with no sensitivity analysis run.

Trains are modelled as a smooth departure rate, with no timetable, dwell variability or disruptions, and every train is assumed to leave with its full 1,000 pax capacity available, even though post-match services through Jolimont and Richmond start at Flinders Street or the City Loop and may already be partly loaded. Each station is also treated as a single pooled queue, so any fan can board any departing train. This overstates throughput most at Richmond, where around ten lines diverge and a fan bound for one line cannot board another line's train. Finally, all fans leave at the final siren with the same arrival shape, and density is averaged over the whole station footprint, ignoring early leavers and crowding concentrated at particular platforms or entrances.

The gate throttle also carries a less obvious assumption. Fans held outside the gate add nothing to platform density and wait in an unbounded holding area at no cost, while boarding efficiency falls continuously as the queue grows, so throttling entry can only ever help clearance in the model. The finding that widening Jolimont's gates makes clearance worse therefore follows largely from this structure, and is better read as a consequence of the model's assumptions than as independent evidence about real gate operations.

The comparative finding, that Jolimont's loading decides whether a fixture clears, is the more useful result, though in practice it reduces to a single team: Collingwood is the only supporter base routed heavily enough through Jolimont to trigger it. Metro Trains could schedule extra Jolimont-line services only when Collingwood plays. Lifting Jolimont from 0.4 to at most 0.55 trains/min clears every Collingwood fixture within the window, while the other 21 fixtures need no change at all. This is far cheaper than boosting frequency across all three corridors for every match, and, within the model's assumptions, more effective than widening Jolimont's gates.

## Conclusion

This report finalised a three-corridor ODE model of MCG post-match rail egress, fixing the dimensional, parameter and inflow inconsistencies from earlier drafts, and tested it across all 28 pairings of the eight parameterised teams. Across these pairings, Jolimont's small relative capacity decides whether the crowd clears within the window. In practice this is a Collingwood effect, and adding Jolimont services for Collingwood fixtures alone would be enough to clear every crowd tested. Future work could validate parameters against real post-match clearance data from Metro Trains, vary attendance and run a sensitivity analysis across parameter ranges, or model cross-corridor rerouting explicitly.

\newpage

## References

[1] Fruin, J. (1971). *Pedestrian Planning and Design*. Metropolitan Association of Urban Designers and Environmental Planners.

[2] Ritz, C., Baty, F., Streibig, J. C., & Gerhard, D. (2015). *Dose-Response Analysis Using R*. PLOS ONE, 10(12), e0146021. Available at: [journals.plos.org](https://journals.plos.org/plosone/article/file?id=10.1371/journal.pone.0146021&type=printable)